# 04 Self-Reflection — Prompt 驱动的自我检查

不加新工具、不加新类、不改 ReAct 循环。只改一行 SYSTEM_PROMPT，LLM 的行为就变了。

**学习点**：Reflection 是 ReAct + 正确 prompt 的自然涌现，不是外加机制。

In [7]:
import sys, json, shutil, os
sys.path.insert(0, '..')

from src.agent_framework import Agent
from src.capabilities import PlanManager
from src.capabilities.demo_tools import create_demo_tools

for d in ['agent_memory']:
    if os.path.exists(d):
        shutil.rmtree(d)

print('一切就绪')

一切就绪


## 不加反思的 Agent

和之前的实验完全一样的 Agent。

In [8]:
PROMPT_NO_REFLECT = (
    "你是一个有用的 AI 助手，可以用中文回复。"
    "需要时使用工具获取信息。"
    "当面对需要多步协调的复杂任务时，先调用 make_plan 制定计划。"
)

plan_mgr = PlanManager()
tools = create_demo_tools(plan_mgr=plan_mgr, ltm=None)
agent_no_reflect = Agent(tools=tools, system_prompt=PROMPT_NO_REFLECT, plan_mgr=plan_mgr)
print('Agent (无反思) 已就绪')

Agent (无反思) 已就绪


### 执行复杂任务

LLM 拿到第一个结果后很可能直接输出答案，不会自检。

In [9]:
reply = agent_no_reflect.chat(
    '特斯拉的股价在当前市场环境下是否值得投资？请综合考虑股价、行业前景和风险因素。'
)
print(f'\n=== 最终回答 ===\n{reply[:500]}...' if len(reply) > 500 else f'\n=== 最终回答 ===\n{reply}')

🔧 [make_plan] → 计划已创建（已保存到文件）：
  1. 搜索特斯拉当前股价和市场表现
  2. 搜索特斯拉最新财报和基本面数据
  3. 搜索电动汽车行业前景分析
  4. 搜索特斯拉面临的风险因素
  5. 综合以上信息给出投资建议

请按顺序执行第一步。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 未找到关于 'Tesla stock price today 2025' 的结果
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [complete_step] → 步骤已完成。（当前第 2 步）
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 未找到关于 'Tesla Q1 2025 earnings revenue EPS' 的结果
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 未找到关于 'Tesla 2024 annual report revenue net income delivery' 的结果
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [complete_step] → 步骤已完成。（当前第 3 步）
🔧 [search_web] → 未找到关于 '电动汽车行业前景 2025 全球电动车市场增长预测' 的结果
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 未找到关于 '2025年 电动车 市场趋势 渗透率 增长' 的结果
🔧 [search_web] → 未找到关于 'globa

## 加了反思的 Agent

同样一套代码，只多了一行 prompt。

In [10]:
PROMPT_WITH_REFLECT = (
    "你是一个有用的 AI 助手，可以用中文回复。"
    "需要时使用工具获取信息。"
    "当面对需要多步协调的复杂任务时，先调用 make_plan 制定计划。"
    "在给出最终答案前，请先自我检查：数据是否准确？逻辑是否完整？是否有遗漏？如果发现问题，先修正再回答。"
)

plan_mgr2 = PlanManager()
tools2 = create_demo_tools(plan_mgr=plan_mgr2, ltm=None)
agent_with_reflect = Agent(tools=tools2, system_prompt=PROMPT_WITH_REFLECT, plan_mgr=plan_mgr2)
print('Agent (有反思) 已就绪')

Agent (有反思) 已就绪


### 同一个任务

对比 LLM 的行为差异。

In [11]:
reply = agent_with_reflect.chat(
    '特斯拉的股价在当前市场环境下是否值得投资？请综合考虑股价、行业前景和风险因素。'
)
print(f'\n=== 最终回答 ===\n{reply[:500]}...' if len(reply) > 500 else f'\n=== 最终回答 ===\n{reply}')

🔧 [make_plan] → 计划已创建（已保存到文件）：
  1. 查询特斯拉当前股价及近期表现
  2. 查询特斯拉最新财报和基本面数据
  3. 查询电动车行业前景及竞争格局
  4. 查询特斯拉面临的风险因素（政策、竞争、估值等）
  5. 综合分析并给出投资建议

请按顺序执行第一步。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 未找到关于 'TSLA stock price today 2025年最新' 的结果
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [complete_step] → 步骤已完成。（当前第 2 步）
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 未找到关于 'Tesla 2024 Q4 earnings revenue EPS financial results' 的结果
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 未找到关于 'Tesla 2024 delivery numbers annual report' 的结果
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 未找到关于 'Tesla Inc financial results 2024 revenue profit' 的结果
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [search_web] → 特斯拉当前股价 $245，上季度为 $220。
🔧 [complete_step] → 步骤已完成。（当前第 3 步）
🔧 [search_web] → 未找到关于 '电动车行业 2025年 前